# Data

In [1]:
data_AE = []

count = 0



with open("/kaggle/input/ilm-data/publish_data.txt", "rb") as f:

    for line in f:

        temp = line.rstrip(b'\r\n').split(b'\x01')

        for i in range(len(temp)):

            temp[i] = temp[i].decode('utf-8')

        while len(temp) < 3:

            count += 1

            temp.append(b'NULL')

        if not data_AE or data_AE[-1][0] != temp[0]:

            data_AE.append([temp[0], {temp[1]: temp[2]}])

        else:

            data_AE[-1][1][temp[1]] = temp[2]



data_AE[0]

['APG STO0045  Camping Stove Portable Cooking Equipment Welding BBQ Butane Hiking Camping Gas Burners Gas Adapter Torch Lighter',
 {'Fuel': 'Gas', 'Brand Name': 'APG', 'Model': 'NULL', 'Disposable': 'NULL'}]

In [2]:
import os

import json



data_oamine = []

directory = "/kaggle/input/ilm-data/annotations/annotations"



for filename in os.listdir(directory):

      filepath = os.path.join(directory, filename)

      with open(filepath, "r") as f:

          for line in f:

              temp = json.loads(line)

              store = {}

              for i in range(len(temp["entities"])):

                  store[temp["entities"][i]["label"]] = temp["entities"][i]["value"]

              data_oamine.append([temp["title"], store])



data_oamine[0]

["Cameron's Coffee Single Serve Pods, Flavored, Chocolate Caramel Brownie, 12 Count (Pack of 1)",
 {'Brand': "Cameron's Coffee",
  'Item form': 'Single Serve Pods',
  'Specialty': 'Flavored',
  'Flavor': 'Chocolate Caramel Brownie',
  'Net content': '12 Count',
  'Pack size': 'Pack of 1'}]

# Model Preparation

In [3]:
%%capture

!pip install pip3-autoremove

!pip-autoremove torch torchvision torchaudio -y

!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu121

!pip install unsloth

In [4]:
from unsloth import FastLanguageModel

import torch

max_seq_length = 64 # Choose any! We auto support RoPE Scaling internally!

dtype = torch.float16 # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+

load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.



model, tokenizer = FastLanguageModel.from_pretrained(

    model_name = "unsloth/Meta-Llama-3.1-8B",

    max_seq_length = max_seq_length,

    dtype = dtype,

    load_in_4bit = load_in_4bit,

)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
==((====))==  Unsloth 2024.11.5: Fast Llama patching. Transformers = 4.46.2.
   \\   /|    GPU: Tesla T4. Max memory: 14.741 GB. Platform = Linux.
O^O/ \_/ \    Pytorch: 2.5.1+cu121. CUDA = 7.5. CUDA Toolkit = 12.1.
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [5]:
#Adding LoRA adapters

model = FastLanguageModel.get_peft_model(

    model,

    r = 8,

    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj",

                      "up_proj", "down_proj",],

    lora_alpha = 16,

    lora_dropout = 0, # Supports any, but = 0 is optimized

    bias = "none",    # Supports any, but = "none" is optimized

    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context

    random_state = 3407,

    use_rslora = False,

    loftq_config = None,

)

Unsloth 2024.11.5 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


# Data Preprocessing

In [6]:
import pandas as pd

from sklearn.model_selection import train_test_split



# Split dataset into training and testing sets

train_data, test_data = train_test_split(data_AE, test_size=0.3, random_state=42)



# Define special tokens

mask_token = "<|mask|>"



# Add special tokens to tokenizer

special_tokens = {"additional_special_tokens": ["<|mask|>"]}

tokenizer.add_special_tokens(special_tokens)

model.resize_token_embeddings(len(tokenizer))



def preprocess_for_finetuning(data):

    input_sequences = []

    target_sequences = []



    for row in data:

        C = row[0]  # Product distribution C

        attributes = row[1]  # Dictionary of attributes (A1, A2, ..., An) and values (V1, V2, ..., Vn)



        # Create the input sentence with masked attributes, C only appears once at the start

        attribute_part = " and ".join([f"{key} is {mask_token}" for key in attributes.keys()])

        input_sentence = f"{C}: {attribute_part}"



        # Create the target sequence with actual values and [answer] token

        target_sentence = "<>".join(attributes.values())



        input_sequences.append(input_sentence)

        target_sequences.append(target_sentence)



    return input_sequences, target_sequences



# Preprocess the training data

train_input_sequences, train_target_sequences = preprocess_for_finetuning(train_data)



# Display some preprocessed training data

for input_seq, target_seq in zip(train_input_sequences[:3], train_target_sequences[:3]):

    print(f"Input: {input_seq}")

    print(f"Target: {target_seq}")

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Input: Hearuisavy Women Running Shorts Quick Dry Breathable Sports Running Fitness Beach Shorts Swimming Yoga Corto Pantalon Hot Shorts: Item Type is <|mask|> and Sport Type is <|mask|> and Pattern Type is <|mask|> and Gender is <|mask|>
Target: Shorts<>Running<>NULL<>Women
Input: HEWOLF   Windproof Waterproof  5-8 People  Camping Outdoor Tent for Fishing, Hunting Adventure,Beach Travel  and Family Party: Style is <|mask|> and Model Number is <|mask|> and Season is <|mask|>
Target: Outdoor<>NULL<>NULL
Input: Women Sexy Thong V Swimwear Beach Pants Bikini Swimsuit Bathing Suit indoor swimsuit 2018 Brazilian bikini sexy bikini set thong:  Bathing Suit is <|mask|>
Target:  Bathing Suit


In [7]:
from datasets import Dataset



# Assuming train_input_sequences and train_target_sequences are defined

training_dataset = Dataset.from_dict({

    "input": train_input_sequences,

    "output": train_target_sequences,

})



alpaca_prompt = """### Input:

{}



### Response:

{}"""



EOS_TOKEN = tokenizer.eos_token  # Ensures generation stops at the end of each example



def formatting_prompts_func(examples):

    inputs = examples["input"]

    outputs = examples["output"]

    texts = []

    for input, output in zip(inputs, outputs):

        # Adding EOS_TOKEN to prevent the model from generating indefinitely

        text = alpaca_prompt.format(input, output) + EOS_TOKEN

        texts.append(text)

    return {"text": texts}



# Map the formatting function onto the training_dataset

formatted_dataset = training_dataset.map(formatting_prompts_func, batched=True)

Map:   0%|          | 0/30056 [00:00<?, ? examples/s]

# Fine-Tuning

In [8]:
from trl import SFTTrainer

from transformers import TrainingArguments

from unsloth import is_bfloat16_supported



trainer = SFTTrainer(

    model = model,

    tokenizer = tokenizer,

    train_dataset = formatted_dataset,

    dataset_text_field = "text",

    max_seq_length = max_seq_length,

    dataset_num_proc = 2,

    packing = False,

    args = TrainingArguments(

        per_device_train_batch_size = 8,

        gradient_accumulation_steps = 4,

        warmup_steps = 5,

        num_train_epochs = 1,

        learning_rate = 2e-4,

        fp16 = not is_bfloat16_supported(),

        bf16 = is_bfloat16_supported(),

        logging_steps = 5,

        optim = "adamw_8bit",

        weight_decay = 0.01,

        lr_scheduler_type = "linear",

        seed = 3407,

        output_dir = "outputs",

        report_to = "none", # Use this for WandB etc

    ),

)

Map (num_proc=2):   0%|          | 0/30056 [00:00<?, ? examples/s]

In [9]:
#@title Show current memory stats

gpu_stats = torch.cuda.get_device_properties(0)

start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)

max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")

print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
12.84 GB of memory reserved.


In [10]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 30,056 | Num Epochs = 1
O^O/ \_/ \    Batch size per device = 8 | Gradient Accumulation steps = 4
\        /    Total batch size = 32 | Total steps = 939
 "-____-"     Number of trainable parameters = 20,971,520


Step,Training Loss
5,5.214000
10,4.041500
15,3.384300
20,3.147300
25,3.002400
30,2.855300
35,2.818300
40,2.744400
45,2.691100
50,2.703900


/opt/conda/lib/python3.10/site-packages/peft/utils/save_and_load.py:257: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/peft/utils/save_and_load.py:257: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


In [11]:
#@title Show final memory and time stats

used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)

used_memory_for_lora = round(used_memory - start_gpu_memory, 3)

used_percentage = round(used_memory         /max_memory*100, 3)

lora_percentage = round(used_memory_for_lora/max_memory*100, 3)

print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")

print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")

print(f"Peak reserved memory = {used_memory} GB.")

print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")

print(f"Peak reserved memory % of max memory = {used_percentage} %.")

print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

7006.1296 seconds used for training.
116.77 minutes used for training.
Peak reserved memory = 12.84 GB.
Peak reserved memory for training = 0.0 GB.
Peak reserved memory % of max memory = 87.104 %.
Peak reserved memory for training % of max memory = 0.0 %.


# Inference

In [12]:
FastLanguageModel.for_inference(model)



def prepare_test_input(data):

    input_sequences = []

    ground_truth_values = []

    repeated_sequences = []  # List to store repeated input sequences

    attribute_list = []  # List to store attributes



    for row in data:

        C = row[0]  # Product distribution C

        attributes = row[1]  # Dictionary of attributes (A1, A2, ..., An) and values (V1, V2, ..., Vn)



        # Create the input sentence with masked attributes, C only appears once at the start

        attribute_part = " and ".join([f"{key} is {mask_token}" for key in attributes.keys()])

        input_sentence = f"{C}: {attribute_part}"

        input_sequences.append(input_sentence)



        # Repeat the input sequence for the number of attribute-value pairs

        repeated_sequences.extend([input_sentence] * len(attributes))



        # Collect the attributes for later use

        attribute_list += list(attributes.keys())



        # Collect only the attribute values for ground truth

        ground_truth_values += list(attributes.values())



    return input_sequences, repeated_sequences, attribute_list, ground_truth_values

# Inference on AE-110k

In [13]:
# Get test inputs and ground truth values

test_input_sequences, repeated_inputs, attribute_list, ground_truth_values = prepare_test_input(test_data)



# Predict values for each test input

predicted_values = []

for idx, input_seq in enumerate(test_input_sequences):

    if (idx+1)%500==0:

        print(f"Processing {idx+1}th example...")

    inputs = tokenizer(

        [

            alpaca_prompt.format(

                input_seq,  # input

                "",  # output - leave this blank for generation!

            )

        ],

        return_tensors="pt"

    ).to("cuda")



    outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True)

    predicted_value = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]



    # Extract the response part from the predicted value

    response_start = predicted_value.find("### Response:")  # Find the start of the response section

    if response_start != -1:

        # Extract the text after the response header

        response = predicted_value[response_start + len("### Response:"):].strip()

    else:

        response = ""  # If not found, set response to an empty string

    values_list = response.split("<>")



    # Ensure values_list size matches the number of mask tokens (attributes) in input_seq

    num_attributes = input_seq.count("<|mask|>")

    if len(values_list) < num_attributes:

        values_list += ['NULL'] * (num_attributes - len(values_list))

    elif len(values_list) > num_attributes:

        values_list = values_list[:num_attributes]



    predicted_values += values_list

Processing 500th example...
Processing 1000th example...
Processing 1500th example...
Processing 2000th example...
Processing 2500th example...
Processing 3000th example...
Processing 3500th example...
Processing 4000th example...
Processing 4500th example...
Processing 5000th example...
Processing 5500th example...
Processing 6000th example...
Processing 6500th example...
Processing 7000th example...
Processing 7500th example...
Processing 8000th example...
Processing 8500th example...
Processing 9000th example...
Processing 9500th example...
Processing 10000th example...
Processing 10500th example...
Processing 11000th example...
Processing 11500th example...
Processing 12000th example...
Processing 12500th example...


In [14]:
import csv



# Define the output file name

output_file = "/kaggle/working/prediction_results_ae.csv"



# Open the file in write mode

with open(output_file, mode="a", newline="") as file:

    writer = csv.writer(file)



    # Write the header row

    writer.writerow(["Product Description", "Attribute List", "Predicted Values", "Ground Truth Values"])



    # Loop through results and save them in the file

    for i in range(len(predicted_values)):

        # Extract only the product description from `repeated_inputs[i]`

        product_description = repeated_inputs[i].split(":")[0].strip()



        # Write each prediction result as a row

        writer.writerow([

            product_description,

            attribute_list[i],

            predicted_values[i],

            ground_truth_values[i]

        ])



print(f"Results successfully saved to {output_file}.")

Results successfully saved to /kaggle/working/prediction_results_ae.csv.


# Inference on OA-Mine

In [15]:
# Prepare test data using the function defined above

test_input_sequences_oamine, repeated_inputs_oamine, attribute_list_oamine, ground_truth_values_oamine = prepare_test_input(data_oamine)



# Predict values for each test input

predicted_values_oamine = []

for idx,input_seq in enumerate(test_input_sequences_oamine):

    if (idx+1)%500==0:

        print(f"Processed {idx+1}th example...")

    inputs = tokenizer(

        [

            alpaca_prompt.format(

                input_seq,  # input

                "",  # output - leave this blank for generation!

            )

        ],

        return_tensors="pt"

    ).to("cuda")



    outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True)

    predicted_value = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]



    # Extract the response part from the predicted value

    response_start = predicted_value.find("### Response:")

    if response_start != -1:

        response = predicted_value[response_start + len("### Response:"):].strip()

    else:

        response = ""

    values_list = response.split("<>")

    

    # Ensure values_list size matches the number of mask tokens (attributes) in input_seq

    num_attributes = input_seq.count("<|mask|>")

    if len(values_list) < num_attributes:

        values_list += ['NULL'] * (num_attributes - len(values_list))

    elif len(values_list) > num_attributes:

        values_list = values_list[:num_attributes]



    predicted_values_oamine += values_list

Processed 500th example...
Processed 1000th example...
Processed 1500th example...


In [16]:
import csv



# Define the output file name

output_file = "/kaggle/working/prediction_results_oa.csv"



# Open the file in write mode

with open(output_file, mode="a", newline="") as file:

    writer = csv.writer(file)



    # Write the header row

    writer.writerow(["Product Description", "Attribute List", "Predicted Values", "Ground Truth Values"])



    # Loop through results and save them in the file

    for i in range(len(predicted_values_oamine)):

        # Extract only the product description from `repeated_inputs[i]`

        product_description = repeated_inputs_oamine[i].split(":")[0].strip()



        # Write each prediction result as a row

        writer.writerow([

            product_description,

            attribute_list_oamine[i],

            predicted_values_oamine[i],

            ground_truth_values_oamine[i]

        ])



print(f"Results successfully saved to {output_file}.")

Results successfully saved to /kaggle/working/prediction_results_oa.csv.


# Evaluation Metrics

In [7]:
from typing import List, Union

from collections import Counter



class EvaluationMetrics:



    @staticmethod

    def exact_match(ground_truth: Union[str, List[str]], predicted: Union[str, List[str]]) -> float:

        if isinstance(ground_truth, list) and isinstance(predicted, list):

            scores = [1.0 if g == p else 0.0 for g, p in zip(ground_truth, predicted)]

            return sum(scores) / len(scores) if scores else 0.0

        return 1.0 if ground_truth == predicted else 0.0



    @staticmethod

    def bow_precision(ground_truth: Union[str, List[str]], predicted: Union[str, List[str]]) -> float:

        def single_precision(gt: str, pred: str) -> float:

            gt_words = Counter(gt.split())

            pred_words = Counter(pred.split())

            common = sum(min(gt_words[word], pred_words[word]) for word in pred_words)

            return common / sum(pred_words.values()) if pred_words else 0.0



        if isinstance(ground_truth, list) and isinstance(predicted, list):

            scores = [single_precision(g, p) for g, p in zip(ground_truth, predicted)]

            return sum(scores) / len(scores) if scores else 0.0

        return single_precision(ground_truth, predicted)



    @staticmethod

    def bow_recall(ground_truth: Union[str, List[str]], predicted: Union[str, List[str]]) -> float:

        def single_recall(gt: str, pred: str) -> float:

            gt_words = Counter(gt.split())

            pred_words = Counter(pred.split())

            common = sum(min(gt_words[word], pred_words[word]) for word in gt_words)

            return common / sum(gt_words.values()) if gt_words else 0.0



        if isinstance(ground_truth, list) and isinstance(predicted, list):

            scores = [single_recall(g, p) for g, p in zip(ground_truth, predicted)]

            return sum(scores) / len(scores) if scores else 0.0

        return single_recall(ground_truth, predicted)



    @staticmethod

    def bow_f1(ground_truth: Union[str, List[str]], predicted: Union[str, List[str]]) -> float:

        precision = EvaluationMetrics.bow_precision(ground_truth, predicted)

        recall = EvaluationMetrics.bow_recall(ground_truth, predicted)

        if precision + recall == 0:

            return 0.0

        return 2 * (precision * recall) / (precision + recall)

In [8]:
metrics = EvaluationMetrics()



ground_truth = "the cat on the mat"

predicted = "the cat is on the mat"



# Exact match

print(metrics.exact_match(ground_truth, predicted))  # Output: 0.0



# Bag of Words Precision

print(metrics.bow_precision(ground_truth, predicted))  # Output: 0.833



# Bag of Words Recall

print(metrics.bow_recall(ground_truth, predicted))  # Output: 1



# Bag of Words F1 score

print(metrics.bow_f1(ground_truth, predicted))

0.0
0.8333333333333334
1.0
0.9090909090909091


# Evaluation on AE-110k

In [9]:
import csv



# Define the input file name

input_file = "ILM-outputs/prediction_results_ae.csv"



# Initialize lists to store predicted values and ground truth values

predicted_values = []

ground_truth_values = []



# Read the CSV file and extract the required columns

with open(input_file, mode="r") as file:

    reader = csv.DictReader(file)

    for row in reader:

        predicted_values.append(row["Predicted Values"])

        ground_truth_values.append(row["Ground Truth Values"])



# Exact match

print("Exact Match: ", metrics.exact_match(ground_truth_values, predicted_values))



# Bag of Words Precision

print("Precision: ", metrics.bow_precision(ground_truth_values, predicted_values))



# Bag of Words Recall

print("Recall: ", metrics.bow_recall(ground_truth_values, predicted_values))



# Bag of Words F1 score

print("F1 score: ", metrics.bow_f1(ground_truth_values, predicted_values))

Exact Match:  0.782736383178419
Precision:  0.8100258912775712
Recall:  0.808231375653186
F1 score:  0.8091276384794786


In [10]:
import csv

from typing import Dict



def evaluate_attribute_from_file(file_path: str, attribute_name: str, metrics) -> Dict[str, float]:

    # Initialize lists for filtered ground truth and predictions

    filtered_ground_truth = []

    filtered_predictions = []



    # Open and read the file

    with open(file_path, mode="r") as file:

        reader = csv.DictReader(file)

        for row in reader:

            # Check if the current row's attribute matches the specified attribute_name

            if row["Attribute List"] == attribute_name:

                filtered_ground_truth.append(row["Ground Truth Values"])

                filtered_predictions.append(row["Predicted Values"])



    # Calculate the metrics for the filtered data

    exact_match = metrics.exact_match(filtered_ground_truth, filtered_predictions)

    precision = metrics.bow_precision(filtered_ground_truth, filtered_predictions)

    recall = metrics.bow_recall(filtered_ground_truth, filtered_predictions)

    f1 = metrics.bow_f1(filtered_ground_truth, filtered_predictions)



    # Store the results in a dictionary

    results = {

        "Exact Match": exact_match,

        "Precision": precision,

        "Recall": recall,

        "F1 Score": f1

    }



    return results

In [11]:
# List of attributes to evaluate

attributes_to_evaluate = [

    "Brand Name", "Material", "Color", "Category", "Frame Color", 
    "Lenses Color", "Shell Material", "Wheel Material", "Product Type"

]



# Evaluate and print results for each attribute

for attribute_name in attributes_to_evaluate:

    results = evaluate_attribute_from_file(input_file, attribute_name, metrics)

    print(f"Results for attribute: {attribute_name}")

    for metric, value in results.items():

        print(f"{metric}: {value}")

    print("-" * 40)  # Separator for readability

Results for attribute: Brand Name
Exact Match: 0.9405624818788054
Precision: 0.9439209432685803
Recall: 0.9438484584903837
F1 Score: 0.9438846994878811
----------------------------------------
Results for attribute: Material
Exact Match: 0.8448729184925504
Precision: 0.8805141688577273
Recall: 0.8791557113643003
F1 Score: 0.8798344157493758
----------------------------------------
Results for attribute: Color
Exact Match: 0.7336065573770492
Precision: 0.7814207650273224
Recall: 0.7762295081967213
F1 Score: 0.7788164860349494
----------------------------------------
Results for attribute: Category
Exact Match: 0.5882352941176471
Precision: 0.6627995642701524
Recall: 0.6519607843137254
F1 Score: 0.6573354972711151
----------------------------------------
Results for attribute: Frame Color
Exact Match: 0.8888888888888888
Precision: 0.8888888888888888
Recall: 0.8888888888888888
F1 Score: 0.8888888888888888
----------------------------------------
Results for attribute: Lenses Color
Exact M

# Evaluation on OA-Mine

In [12]:
import csv



# Define the input file name

input_file = "ILM-outputs/prediction_results_oa.csv"



# Initialize lists to store predicted values and ground truth values

predicted_values_oamine = []

ground_truth_values_oamine = []



# Read the CSV file and extract the required columns

with open(input_file, mode="r") as file:

    reader = csv.DictReader(file)

    for row in reader:

        predicted_values_oamine.append(row["Predicted Values"])

        ground_truth_values_oamine.append(row["Ground Truth Values"])



# Exact match

print("Exact Match:", metrics.exact_match(ground_truth_values_oamine, predicted_values_oamine))



# Bag of Words Precision

print("Precision:", metrics.bow_precision(ground_truth_values_oamine, predicted_values_oamine))



# Bag of Words Recall

print("Recall:", metrics.bow_recall(ground_truth_values_oamine, predicted_values_oamine))



# Bag of Words F1 score

print("F1 score:", metrics.bow_f1(ground_truth_values_oamine, predicted_values_oamine))

Exact Match: 0.4265620222199572
Precision: 0.5852354319878015
Recall: 0.5494386371409872
F1 score: 0.5667723743874464
